# Quantitative central-limit theorem

This notebook generates `fig:matching-quantitative-clt`.  Starting from the symmetric Bernoulli law
$$
\alpha_0=\frac12(\delta_{-1}+\delta_{1}),
$$
we display the law of the normalized sum
$$
Z_n=\frac{1}{\sqrt n}\sum_{i=1}^n X_i,
\qquad X_i\sim \alpha_0,
$$
for increasing values of $n$.  The panels are deliberately elementary: finite Dirac masses are drawn as small circular atoms and vertical stems, while the gray curve is the limiting standard Gaussian density.

In [1]:

from pathlib import Path
import os
import sys

os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/mpl-ot4ml")

for candidate in [Path.cwd(), Path.cwd() / "notebooks-figures", Path.cwd().parent / "notebooks-figures"]:
    if (candidate / "figure_style.py").exists():
        sys.path.insert(0, str(candidate.resolve()))
        break
else:
    raise RuntimeError("Could not locate figure_style.py")

import matplotlib.pyplot as plt
import numpy as np
import ot
from math import comb

from figure_style import (
    BLUE,
    RED,
    VIOLET,
    ORANGE,
    GRAY,
    LIGHT_GRAY,
    figure_dir,
    box_axes,
    remove_axes,
    save_pdf,
    setup_matplotlib,
    interp_color,
)

setup_matplotlib()

fig_name = "matching-quantitative-clt"
out = figure_dir(fig_name)


## Bernoulli sums

For $k=0,\ldots,n$, the normalized sum takes the value $(2k-n)/\sqrt n$ with mass $2^{-n}\binom{n}{k}$.  For the largest panel this still has only $n+1$ atoms, so the notebook is exact and lightweight.

In [2]:

def gaussian_pdf(x):
    return np.exp(-0.5 * x**2) / np.sqrt(2 * np.pi)


def bernoulli_sum_law(n):
    k = np.arange(n + 1)
    x = (2 * k - n) / np.sqrt(n)
    # Normalize after exponentiating logs; this stays stable for large n.
    logw = np.array([np.log(comb(n, int(ki))) for ki in k], dtype=float) - n * np.log(2.0)
    w = np.exp(logw - logw.max())
    w = w / w.sum()
    return x, w


def clt_panel(n, filename, color):
    x, w = bernoulli_sum_law(n)
    grid = np.linspace(-3.8, 3.8, 600)
    g = gaussian_pdf(grid)
    spacing = 2 / np.sqrt(n)
    heights = w / spacing

    fig, ax = plt.subplots(figsize=(1.55, 1.38))
    ax.plot(grid, g, color=GRAY, lw=1.05, alpha=0.82, zorder=1)
    for xi, hi in zip(x, heights):
        ax.plot([xi, xi], [0, hi], color=color, lw=0.55, alpha=0.58, zorder=2)
    sizes = 6.0 + 130 * (heights / heights.max()) ** 0.85
    ax.scatter(x, heights, s=sizes, marker='o', color=color, edgecolor='none', alpha=0.88, zorder=3)
    ax.set_xlim(-3.35, 3.35)
    ax.set_ylim(-0.015, 0.46)
    ax.set_xticks([-2, 0, 2])
    ax.set_yticks([])
    ax.tick_params(axis='x', labelsize=6, pad=1)
    box_axes(ax)
    save_pdf(fig, out / filename, pad_inches=0.025)
    plt.close(fig)


## Exported panels

The color is interpolated from red to blue as $n$ increases.  No title is embedded in the PDFs; the LaTeX layout supplies the values of $n$.

In [3]:

ns = [1, 2, 4, 16, 64]
for t, n in enumerate(ns):
    clt_panel(n, f"n-{n:03d}.pdf", interp_color(t / (len(ns) - 1)))
